2 Build Feed forward neural network using different Regularization techniques

In [1]:
!pip install -q torchvision



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Prepare Data (MNIST)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

# 2. Define the Regularized Model
class RegularizedNet(nn.Module):
    def __init__(self, use_dropout=False, use_batchnorm=False):
        super(RegularizedNet, self).__init__()
        self.flatten = nn.Flatten()
        
        # Hidden Layer 1
        self.fc1 = nn.Linear(28*28, 256)
        # self.bn1 = nn.BatchNormalization1d(256) if use_batchnorm else nn.Identity()
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) if use_dropout else nn.Identity()
        
        # Hidden Layer 2
        self.fc2 = nn.Linear(256, 128)
        
        # Output Layer
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# 3. Training Function
def train_model(model, name, weight_decay=0.0):
    print(f"\n--- Training {name} ---")
    criterion = nn.CrossEntropyLoss()
    # weight_decay > 0 applies L2 Regularization
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    
    model.train()
    for epoch in range(2): # 2 epochs for quick demonstration
        running_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Final Accuracy of {name}: {100 * correct / total:.2f}%")

# 4. Run Comparisons
# Baseline Model
train_model(RegularizedNet(), "Baseline Model")

# Model with Dropout & L2 Regularization
train_model(RegularizedNet(use_dropout=True), "Dropout + L2 Model", weight_decay=1e-4)

# Model with Batch Normalization
train_model(RegularizedNet(use_batchnorm=True), "Batch Norm Model")


--- Training Baseline Model ---
Epoch 1, Loss: 0.2203
Epoch 2, Loss: 0.0935
Final Accuracy of Baseline Model: 97.37%

--- Training Dropout + L2 Model ---
Epoch 1, Loss: 0.2639
Epoch 2, Loss: 0.1371
Final Accuracy of Dropout + L2 Model: 96.99%

--- Training Batch Norm Model ---
Epoch 1, Loss: 0.2194
Epoch 2, Loss: 0.0938
Final Accuracy of Batch Norm Model: 97.08%
